# CLaS-Bench: DiffMean Layer-by-Layer Language Steering Evaluation

This notebook executes the canonical CLaS-Bench scripts for **DiffMean** representation steering on `CohereLabs/aya-expanse-8b` across representative layers (`4, 12, 20, 28`) on 5 languages spanning the resource & script spectrum: English (`en`), German (`de`), Czech (`cs`), Maltese (`mt`), and Tibetan (`bo`).

### Pipeline:
1. **Setup & Clone**: Prepares the repository.
2. **Dependencies**: Installs `vllm`, `transformers<5.0.0`, `bitsandbytes`, and FastText evaluation tools.
3. **Authentication**: Logs into Hugging Face for model access.
4. **Intervention**: Runs `2-diffmean-4.sh` (Aya Expanse DiffMean runner across layers 4, 12, 20, 28 with 15 questions and max_tokens=300).
5. **Evaluation**: Runs `X-evaluate-language-forcing-diffmean.sh` (FastText forcing success).
6. **Visualization**: Runs `B-vis-ablate-diffmean.sh` (generates multi-layer comparison plots and heatmaps).
7. **Download**: Archives results into a zip file for download.


In [1]:
# 1. Environment & Repository Setup
import os

if not os.path.exists('/content/CLaS-Bench'):
    !git clone https://github.com/Trier-CLaS-SemEval-2027/CLaS-Bench.git /content/CLaS-Bench

%cd /content/CLaS-Bench/CLaS-Bench
!git checkout exploration
!git pull origin exploration
!pwd


Cloning into '/content/CLaS-Bench'...
remote: Enumerating objects: 550, done.
remote: Counting objects: 100% (100/100), done.
remote: Compressing objects: 100% (72/72), done.
remote: Total 550 (delta 53), reused 65 (delta 26), pack-reused 450 (from 2)
Receiving objects: 100% (550/550), 341.18 MiB | 20.06 MiB/s, done.
Resolving deltas: 100% (113/113), done.
Updating files: 100% (439/439), done.
/content/CLaS-Bench/CLaS-Bench
branch 'exploration' set up to track 'origin/exploration'.
Switched to a new branch 'exploration'
From https://github.com/Trier-CLaS-SemEval-2027/CLaS-Bench
 * branch            exploration -> FETCH_HEAD
Already up to date.
/content/CLaS-Bench/CLaS-Bench


In [ ]:
# 2. Install dependencies (vllm + 4-bit bitsandbytes + fasttext for evaluation)
!pip install -q "vllm==0.10.2" "transformers<5.0.0" bitsandbytes datasets fasttext-numpy2-wheel seaborn matplotlib

""" In case you see "ERROR: pip's dependency resolver does not currently take into account
 all the packages that are installed. This behaviour is the source of the following dependency conflicts."
 
 Just ignore it. Are just non-fatal warnings for our purpose"


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [3]:
# 3. Check GPU & Authenticate with Hugging Face
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

from huggingface_hub import login
login()


PyTorch version: 2.8.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
# 4. Run DiffMean Steering Intervention for Aya-Expanse (2-diffmean-4.sh)
# Sweeps layers 4, 12, 20, 28 across en, de, cs, mt, bo with 15 questions and max_tokens=300 in 4-bit quantization
%cd /content/CLaS-Bench/CLaS-Bench
!bash 2-diffmean-4.sh


In [ ]:
# 5. Evaluate Language Forcing Success Rate (X-evaluate-language-forcing-diffmean.sh)
%cd /content/CLaS-Bench/CLaS-Bench
!bash X-evaluate-language-forcing-diffmean.sh


In [ ]:
# 6. Visualize Layer-by-Layer DiffMean Results (B-vis-ablate-diffmean.sh)
%cd /content/CLaS-Bench/CLaS-Bench
!bash B-vis-ablate-diffmean.sh

from IPython.display import Image, display
import glob

# Display generated plots inline
plots = sorted(glob.glob("generation/diffmean/*.png"))
for p in plots:
    print(f"\nDisplaying: {p}")
    display(Image(filename=p))


In [ ]:
# 7. Package and Download Generated Results
!zip -r /content/generation_diffmean_layers.zip generation/

from google.colab import files
try:
    files.download('/content/generation_diffmean_layers.zip')
    print("Download initiated in browser!")
except Exception as e:
    print("Tip: If running in VS Code / IDE, right-click 'generation_diffmean_layers.zip' in the Colab files sidebar to download.")
